In [ ]:
import os
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:
data_dir = "data/DHRD/data_se"
orders_df = pd.read_csv(os.path.join(data_dir, "orders_se.txt"))
vendors_df = pd.read_csv(os.path.join(data_dir, "vendors_se.txt"))

print(f"Order rows: {len(orders_df):,}, vendors: {len(vendors_df):,}")

In [ ]:
orders_df['days_offset'] = orders_df['order_day'].str.extract(r'(\d+)').astype(int)

base_date = pd.Timestamp('2023-01-01')
orders_df['order_date'] = base_date + pd.to_timedelta(orders_df['days_offset'], unit='days')

orders_df['order_time'] = pd.to_datetime(orders_df['order_time'], format='%H:%M:%S').dt.time

orders_df['order_datetime'] = pd.to_datetime(
    orders_df['order_date'].dt.strftime('%Y-%m-%d') + ' ' +
    orders_df['order_time'].astype(str)
)

In [ ]:
min_total_orders = 20
min_unique_customers = 5
min_active_days = 3

vendor_order_counts = orders_df.groupby('vendor_id').agg(
    total_orders=('order_id', 'nunique'),
    unique_customers=('customer_id', 'nunique'),
    first_order_date=('order_date', 'min'),
    last_order_date=('order_date', 'max'),
).reset_index()

vendor_order_counts['active_days'] = (vendor_order_counts['last_order_date'] -
                                      vendor_order_counts['first_order_date']).dt.days + 1

qualified_vendors = vendor_order_counts[
    (vendor_order_counts['total_orders'] >= min_total_orders) &
    (vendor_order_counts['unique_customers'] >= min_unique_customers) &
    (vendor_order_counts['active_days'] >= min_active_days)
]
qualified_vendor_ids = set(qualified_vendors['vendor_id'])

filtered_orders = orders_df[orders_df['vendor_id'].isin(qualified_vendor_ids)].copy()

print(f"Vendors: {len(vendor_order_counts):,} -> {len(qualified_vendor_ids):,}")
print(f"Order rows: {len(orders_df):,} -> {len(filtered_orders):,}")

In [ ]:
choice_situations = filtered_orders.groupby(['customer_id', 'order_id', 'vendor_id']).agg({
    'geohash': 'first',
    'order_datetime': 'first',
    'days_offset': 'first',
}).reset_index()

choice_situations.columns = ['customer_id', 'order_id', 'chosen_vendor_id', 'customer_geohash',
                             'order_datetime', 'days_offset']

In [ ]:
vendors_by_geohash = defaultdict(set)
for geohash, vendor_id in zip(filtered_orders['geohash'], filtered_orders['vendor_id']):
    vendors_by_geohash[geohash].add(vendor_id)

vendors_active_by_day = defaultdict(set)
for days_offset, vendor_id in filtered_orders.groupby(['days_offset', 'vendor_id']).size().index:
    vendors_active_by_day[days_offset].add(vendor_id)

In [ ]:
def get_available_vendors(customer_geohash, order_day):
    geohash_vendors = vendors_by_geohash.get(customer_geohash, set())
    day_vendors = vendors_active_by_day.get(order_day, set())
    return list(geohash_vendors.intersection(day_vendors))

choice_situations['available_vendors'] = choice_situations.apply(
    lambda row: get_available_vendors(row['customer_geohash'], row['days_offset']),
    axis=1
)
choice_situations['num_available_vendors'] = choice_situations['available_vendors'].apply(len)

chosen_available = choice_situations.apply(
    lambda row: row['chosen_vendor_id'] in row['available_vendors'], axis=1
)
choice_situations = choice_situations[chosen_available].copy()

print(f"Choice situations: {len(chosen_available):,}, chosen vendor available: {chosen_available.sum():,}")

In [ ]:
min_options = 2
min_choices_per_customer = 10

filtered_choices = choice_situations[
    (choice_situations['num_available_vendors'] >= min_options)
].copy()

customer_choice_counts = filtered_choices['customer_id'].value_counts()
qualifying_customers = customer_choice_counts[customer_choice_counts >= min_choices_per_customer].index

final_choices = filtered_choices[filtered_choices['customer_id'].isin(qualifying_customers)].copy()

print(f"Choice situations with at least {min_options} options: {len(filtered_choices):,}")
print(f"Customers with at least {min_choices_per_customer} of them: {final_choices['customer_id'].nunique():,} "
      f"({len(final_choices):,} choice situations)")

In [ ]:
customer_encoder = LabelEncoder()
vendor_encoder = LabelEncoder()

all_customers = final_choices['customer_id'].unique()
all_vendors = set()
for vendor_list in final_choices['available_vendors']:
    all_vendors.update(vendor_list)
all_vendors = list(all_vendors)

customer_encoder.fit(all_customers)
vendor_encoder.fit(all_vendors)

final_choices['user_id'] = customer_encoder.transform(final_choices['customer_id'])
final_choices['choice_id'] = vendor_encoder.transform(final_choices['chosen_vendor_id'])
final_choices['option_ids'] = final_choices['available_vendors'].apply(
    lambda vendor_list: vendor_encoder.transform(vendor_list).tolist()
)

In [ ]:
training_data = final_choices[['user_id', 'choice_id', 'option_ids', 'order_datetime']].copy()

max_options = training_data['option_ids'].apply(len).max()
padding_id = -1

training_data['option_ids_fixed'] = training_data['option_ids'].apply(
    lambda option_list: option_list + [padding_id] * (max_options - len(option_list))
)
assert training_data.apply(lambda row: row['choice_id'] in row['option_ids_fixed'], axis=1).all()

training_data = training_data.sort_values(['user_id', 'order_datetime']).reset_index(drop=True)

users = training_data['user_id'].values
choices = training_data['choice_id'].values
options = np.array(training_data['option_ids_fixed'].tolist())

In [ ]:
vendor_info = vendors_df[vendors_df['vendor_id'].isin(all_vendors)].copy()
vendor_info['encoded_vendor_id'] = vendor_encoder.transform(vendor_info['vendor_id'])
vendor_info = vendor_info.sort_values('encoded_vendor_id')

cuisine_dummies = pd.get_dummies(vendor_info['primary_cuisine'], prefix='cuisine')
padding_features = np.zeros((1, cuisine_dummies.shape[1]))
vendor_features = np.vstack([cuisine_dummies.values, padding_features])

In [ ]:
output_dir = "data/DHRD/pre-processed"
os.makedirs(output_dir, exist_ok=True)

np.savez(
    os.path.join(output_dir, "stockholm_restaurants.npz"),
    users=users,
    choices=choices,
    options=options,
    vendor_features=vendor_features,
    padding_id=padding_id,
    no_choice_id=-999,
    max_options=max_options,
    num_users=len(np.unique(users)),
    num_vendors=len(all_vendors),
)

print(f"Choice situations: {len(choices):,}")
print(f"Users: {len(np.unique(users)):,}")
print(f"Vendors: {len(all_vendors):,}")
print(f"Options per choice situation: {(options != padding_id).sum(axis=1).mean():.1f} on average, {max_options} at most")
print(f"Vendor features: {vendor_features.shape[1]} cuisines")